# Path A, P23 — design builder, L1 parameter recovery, target-code timing (Oct 29, 2026)

**Spec (as given; not changed after seeing results):**

TASK: (1) a design builder, (2) parameter recovery for L1, (3) target-code timing.
Do NOT compare simulated targets with human targets. Never load Ring/CT (never pass allow_test).

0. Decisions on the open questions
   - Keep hazard.py's old-CL feature definitions in the simulator: at_end = |k| >= 9; first move = first change of k; tau_r resets at a reversal's first k-step.
   - Keep CC features on the 0.033 s tick snapshot, as built.
   - Write both decisions in sim.py's docstring.
1. Design builder (src/slider/design.py)
   - design(run_set) -> one row per real round: group, dataset (EL/CL/CC), role, topology, round id; graph + seat->node map; interface (lab v09 / old CL / CC); step d; speed; dt = d / speed; time limit T; lab flag; whole_second flag.
   - Speed rule: lab rounds: that round's own speed from interface.py (modal pace x step), falling back to the file's modal speed; CC rounds: the file's cap (the file's modal speed); whole-second old-CL files: 0.2 units/s.
   - Run sets: "L3" = load_runs("outcome", roles={"train","val"}) (targets 1-7); "L5" = load_runs("behavior", roles={"train","val"}, keep_bot=False, keep_incomplete=False) (targets 8-10); "G6train" = load_runs("behavior") (the G-train rounds of the hazard table, bot and incomplete runs included; they are simulated all-human).
   - Checks: report rounds per (topology, dataset) cell for each set; L5 is a subset of L3, and L3 minus L5 = the whole-second rounds only.
   - simulate_design(design, preset, R, seed) -> (runs, table), using each round's own dataset, role, interface, speed and T.
2. Parameter recovery (L1)
   - Simulate L1 on the G6train design, R = 2, seed 573. Report the simulated bins and reversals next to G-train's (336,073 bins; 6,073 reversals).
   - Build a hazard table from the simulated runs with exactly the real pipeline: hazard.build_hazard_table, b = 0.25, m = 2, CC through the 0.1 s lab clock.
   - Refit M2' with fit.fit: R0 structure (shared weights + lab term), all cells. PASS: every weight within 2 model SEs of the value used in the simulation.
   - Re-estimate lam_stop, lam_resume, lam_start per interface on the simulated runs, with the P17 code (motion.py). PASS: each within its 95% CI in results/tables/p17_rates.csv.
   - If exactly one weight misses: rerun once with seed 574 and report both runs.
   - Never change the simulator or the pipeline to make recovery pass. Report any failure as it is.
   - Output: results/tables/p23_recovery.csv with columns true, recovered, SE, z, pass.
3. Target-code timing (planning only)
   - Simulate L1 on the L3 design, R = 1.
   - Time each of these: the simulation; targets.train_targets on the L3 runs; reversal_rate_table, agreement_at_reversal_table and crossing_time_table on the L5 runs.
   - Do NOT print, save or compare any target values.
4. Report: files and tests (the full suite passes); the recovery table and verdicts; the timings; any spec gaps and how they were filled. Commit.

## Choices the spec leaves open

Fixed in `design.py` and this notebook before any recovery number was computed.

- **File speeds** are the P7e values in `results/tables/interface_table.csv`. A lab round's own speed is `interface.step_and_pace(interface.moves(run))`, the same rule as `hazard.build_hazard_table` (its `round_speed`).
- **Seat -> node map**: the topology's graph in `models/apem.GRAPHS`, matched to the round's adjacency with networkx (`design.seat_map`). Any valid map gives the same simulated round.
- **Simulated groups**: copy r of log file g is group `sim<r>_<g>`, with the same rounds (waves) as g. The file name keeps g's version tag, so `interface.py` reads the interface back.
- **Seeds**: `SeedSequence(seed)` is split into one stream per design row, then into a simulation stream and a logger stream.
- **Recovery fit**: `fit.fit(ht, [A, abs_s, at_end, time_at_end_log1p, time_since_rev_log1p], lab_interact=False)` on every simulated bin. Every simulated round copies a G-train round, so no split filter is applied. The weights used in the simulation are `sim.coefs("M2'")`, the exact values in `p11_coefs.csv`. z = (recovered - true) / model SE; PASS if |z| <= 2.
- **Rates**: `motion.rates_table(runs)` as in P17 (1,000 resamples, seed 573), for lab_v09, old_cl and cc; the pooled "shared" row is not used, because the simulator uses per-interface rates. PASS if the estimate lies within the P17 95% CI. The SE and z columns for rates use the Poisson SE sqrt(events) / exposure; they are descriptive, and the verdict uses the CI.
- **Rerun rule**: if exactly one of the 7 weights misses, everything in part 2 is rerun once with seed 574, and both runs go into `p23_recovery.csv` (column `seed`). Otherwise no rerun.
- **Timing**: the L5 runs are the simulated L3 runs of the L5 rounds (L3 minus the whole-second rounds). `train_targets` and the three tables run with their default settings. Results are kept in variables only; nothing is displayed.

In [1]:
import time

import numpy as np
import pandas as pd
import statsmodels.api as sm

from slider import config, design, fit, hazard, motion, sim, targets

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

## 1. Designs: rounds per (topology, dataset) cell

In [2]:
designs = {name: design.design(name) for name in design.SETS}       # about 15 s
cells = pd.concat({name: d.groupby(["topology", "dataset"]).size() for name, d in designs.items()}, axis=1)
cells = cells.fillna(0).astype(int)
cells.loc[("all", ""), :] = cells.sum()
cells

L3     L5  G6train
topology  dataset                       
Full      CC        36.0   36.0     48.0
          CL        63.0   54.0     33.0
Hierarchy CC        40.0   40.0     50.0
          CL        61.0   53.0     33.0
          EL        51.0   51.0     33.0
Star      CC        39.0   39.0     47.0
all                290.0  273.0    244.0

In [3]:
l3, l5 = designs["L3"], designs["L5"]
print("L5 is a subset of L3:", set(l5["round_id"]) <= set(l3["round_id"]))
print("L3 minus L5 = the whole-second rounds:",
      set(l3["round_id"]) - set(l5["round_id"]) == set(l3.loc[l3["whole_second"], "round_id"]),
      f"({int(l3['whole_second'].sum())} rounds)")
print("roles:", {name: sorted(d["role"].unique()) for name, d in designs.items()})

L5 is a subset of L3: True
L3 minus L5 = the whole-second rounds: True (17 rounds)
roles: {'L3': ['train', 'val'], 'L5': ['train', 'val'], 'G6train': ['train']}


Interface, speed (units/s) and dt (s) of the rounds in each set:

In [4]:
pd.concat({name: d.groupby(["interface", "whole_second", "speed", "dt"]).size() for name, d in designs.items()},
          axis=1).fillna(0).astype(int)

L3   L5  G6train
interface whole_second speed dt                         
cc        False        2.94  0.034014    9    9       17
                       3.03  0.033003   97   97      102
                       3.33  0.030030    9    9       17
lab_v09   False        0.50  0.200000    4    4        3
                       1.00  0.100000  102  102       68
                       2.00  0.050000   23   23       16
old_cl    False        0.20  0.100000   29   29       12
          True         0.20  0.100000   17    0        0
cc        False        3.23  0.030960    0    0        9

## 2. Parameter recovery, first run (recorded)

Run on Oct 29 with the simulator at commit `59b7e68` (before the fixes in section 5); the code of that run is in that commit. G6train design, L1, R = 2, seed 573. Simulated 488 rounds: 1,413,764 bins and 34,610 reversals (per copy 706,882 and 17,305; G-train: 336,073 and 6,073). Verdicts: M2' weights 5 of 7 pass (lab z = -3.43, time_since_rev_log1p z = +3.24), rates 6 of 9 pass (lam_stop missed in all three interfaces): FAIL. Two weights missed, so the seed-574 rule (rerun if exactly one misses) did not apply. The table is kept as `results/tables/p23_recovery_v1.csv`:

In [5]:
v1 = pd.read_csv(config.P23_RECOVERY_V1)
with pd.option_context("display.float_format", "{:.4f}".format):
    display(v1)

,seed,part,parameter,true,recovered,SE,z,ci_low,ci_high,pass
0,573,M2' weight,const,-3.1941,-3.1957,0.0252,-0.0637,NaN,NaN,True
1,573,M2' weight,lab,0.0578,0.0179,0.0117,-3.4262,NaN,NaN,False
2,573,M2' weight,A,-2.1074,-2.1089,0.0225,-0.0672,NaN,NaN,True
3,573,M2' weight,abs_s,1.0476,1.0637,0.0382,0.4208,NaN,NaN,True
4,573,M2' weight,at_end,0.4446,0.4033,0.0248,-1.6611,NaN,NaN,True
5,573,M2' weight,time_at_end_log1p,-0.3774,-0.3988,0.0112,-1.9023,NaN,NaN,True
6,573,M2' weight,time_since_rev_log1p,-0.2093,-0.1697,0.0122,3.2399,NaN,NaN,False
7,573,rate,lab_v09 lam_stop,3.0393,2.4806,0.0121,-46.0077,2.4854,3.5327,False
8,573,rate,lab_v09 lam_resume,1.7681,1.6136,0.0079,-19.6151,1.4773,2.0651,True
9,573,rate,lab_v09 lam_start,0.3171,0.3059,0.0088,-1.2799,0.2675,0.3714,True


**Rate conversion (descriptive; added after a smoke test on 3 rounds per interface showed lam_stop low in every interface; it changes no verdict).** The first-run simulator turned each rate into a per-tick probability p = 1 - exp(-lam dt). The P17 estimator counts events per tick of exposure, so it estimates p / dt. From this alone, recovered / true would be (1 - exp(-lam dt)) / (lam dt), at each interface's most common dt.

In [6]:
dts = designs["G6train"].groupby("interface")["dt"].agg(lambda x: x.mode()[0])
r = v1[v1["part"] == "rate"].copy()
r["interface"] = r["parameter"].str.split().str[0]
r["dt (modal)"] = r["interface"].map(dts)
x = r["true"] * r["dt (modal)"]
r["expected ratio"] = (1 - np.exp(-x)) / x
r["observed ratio"] = r["recovered"] / r["true"]
r[["parameter", "true", "recovered", "dt (modal)", "expected ratio", "observed ratio", "pass"]].round(3)

,parameter,true,recovered,dt (modal),expected ratio,observed ratio,pass
7,lab_v09 lam_stop,3.039,2.481,0.100,0.862,0.816,False
8,lab_v09 lam_resume,1.768,1.614,0.100,0.917,0.913,True
9,lab_v09 lam_start,0.317,0.306,0.100,0.984,0.965,True
10,old_cl lam_stop,1.760,1.492,0.100,0.917,0.848,False
11,old_cl lam_resume,1.736,1.704,0.100,0.918,0.982,True
12,old_cl lam_start,0.080,0.085,0.100,0.996,1.065,True
13,cc lam_stop,10.350,8.279,0.033,0.847,0.800,False
14,cc lam_resume,2.549,2.448,0.033,0.959,0.960,True
15,cc lam_start,0.204,0.203,0.033,0.997,0.996,True


## 3. Target-code timing, first run (recorded)

L1 on the L3 design, R = 1, simulator at `59b7e68`; no target value was printed, saved or compared.

| step | seconds |
|---|---|
| simulate L1 on L3, R = 1 (290 rounds) | 139.2 |
| `train_targets` (L3 runs) | 36.5 |
| `reversal_rate_table` (L5 runs) | 4.4 |
| `agreement_at_reversal_table` (L5 runs) | 221.7 |
| `crossing_time_table` (L5 runs) | 29.0 |

Section 7 times the fast path for the check statistics.

## 4. Diagnosis of the first run (recorded)

A traced copy of the first-run loop (same seeds and `sim.tick` calls, simulator at `59b7e68`) recorded each player's mode before and after every tick. It reproduces the first run's rate estimates exactly, and a step-by-step copy of `motion.motion_spells` gives the same totals as `motion.py` on every seat. The tables below come from that traced run (G6train, L1, R = 2, seed 573).

**Rates: the simulator's own counts vs what `motion.py` counts**

| interface | rate | lam (input) | true events | motion.py events | ratio | true at-risk s | motion.py time s | ratio |
|---|---|---|---|---|---|---|---|---|
| lab v09 | stop | 3.039 | 45,461 | 41,728 | 0.918 | 17,312 | 16,822 | 0.972 |
| lab v09 | resume | 1.768 | 42,015 | 42,015 | 1.000 | 26,034 | 26,037 | 1.000 |
| lab v09 | start | 0.317 | 1,218 | 1,218 | 1.000 | 4,097 | 3,982 | 0.972 |
| old CL | stop | 1.760 | 21,542 | 20,866 | 0.969 | 13,430 | 13,986 | 1.041 |
| old CL | resume | 1.736 | 19,986 | 21,162 | 1.059 | 12,413 | 12,417 | 1.000 |
| old CL | start | 0.080 | 168 | 168 | 1.000 | 1,989 | 1,972 | 0.992 |
| CC | stop | 10.350 | 63,795 | 59,483 | 0.932 | 7,262 | 7,185 | 0.990 |
| CC | resume | 2.549 | 60,440 | 60,440 | 1.000 | 24,692 | 24,694 | 1.000 |
| CC | start | 0.204 | 2,030 | 2,030 | 1.000 | 10,067 | 10,002 | 0.993 |

**Recovered / lam vs (1 - exp(-x)) / x, x = lam dt** (the simulator's own rate sits on the formula; resume in lab v09 and CC is fully explained by it)

| | recovered / lam | (1 - e^-x) / x | simulator's own rate / lam |
|---|---|---|---|
| lab v09 stop / resume / start | 0.816 / 0.913 / 0.965 | 0.862 / 0.917 / 0.984 | 0.864 / 0.913 / 0.938 |
| old CL stop / resume / start | 0.848 / 0.982 / 1.065 | 0.917 / 0.918 / 0.996 | 0.911 / 0.928 / 1.056 |
| CC stop / resume / start | 0.800 / 0.960 / 0.996 | 0.847 / 0.959 / 0.997 | 0.849 / 0.960 / 0.990 |

**Stops: `motion.py` pairs each counted moving step with the next tick.** Its stops equal the expected stops on the counted steps that can be followed by a stop:

| interface | next tick moving (eligible) | at the wall | reversing (old CL, before k changes) | round end | motion.py stops | expected on eligible steps |
|---|---|---|---|---|---|---|
| lab v09 | 166,463 (94.29%) | 9,913 (5.62%) | 0 | 161 (0.09%) | 41,728 | 41,451 |
| old CL | 129,778 (92.79%) | 1,366 (0.98%) | 8,669 (6.20%) | 51 (0.04%) | 20,866 | 20,947 |
| CC | 207,775 (94.16%) | 12,625 (5.72%) | 0 | 258 (0.12%) | 59,483 | 59,434 |

**Ticks where the simulator allows an event and `motion.py` does not count it, or the reverse**
- Stop, step onto the wall: `motion.py` counted it as stop exposure, a halt after it is wall time and never a stop (5.6% / 1.0% / 5.7% of counted steps). Against the locked P17 spec: fixed in `motion.py` (F1b).
- Stop, old CL steps of a reversal before k changes: counted as moving time, the first-run simulator could not stop there (6.2%). Fixed by F3.
- Stop on the tick right after a reversal's confirming step (lab 3,733, old CL 676, CC 4,312): never counted, but its exposure is not counted either: neutral.
- Stop, last step before the round end (about 0.1%): no stop possible; negligible.
- Off by one tick: the resume or start step is counted as exposure, the stop tick is not; these cancel.
- Resume, old CL: a reversal from a halt whose first step does not change k is a resume for `motion.py` (+1,176, +5.9%).
- Start: `motion.py`'s start delay leaves out the start tick, so it counts starts per tick before the first step (odds, p / (1 - p)).

**M2' weights**

| | lab v09 | old CL | CC |
|---|---|---|---|
| fires / their expectation (sum of per-tick h), z | 15,278 / 15,265.4, +0.10 | 4,531 / 4,583.9, -0.78 | 15,546 / 15,227.3, **+2.58** |
| detected reversals / y_rev in kept bins | 15,276 / 15,009 | 4,522 / 4,465 | 15,355 / 15,136 |
| at-risk s: simulator / table | 158,694 / 158,812 | 44,195 / 45,078 | 149,451 / 149,551 |
| dating lag of a reversal (detected first step - fire's first step) | 0 for all | mean 0.19 s (to 0.4 s, first step of k) | mean 0.11 s (median 0.05, lab clock) |
| time_since_rev z, refit on this interface only | +2.80 | +2.61 | +2.40 |

- lab (z -3.43): CC fired 2.1% more reversals than their own expectation (chance, z = +2.58) and old CL's table holds 2.0% more at-risk time than the simulator (steps before k changes).
- time_since_rev (z +3.24): too flat in every interface, also lab v09, where dating is exact. Noise-free check in lab v09: the simulator's own expected reversals per table bin over the bin-start model, by time since the last reversal (0-0.25, 0.25-0.5, 0.5-1, 1-2, 2-4, 4-8, 8-16, 16+ s): 0.963, 0.954, 1.004, 1.017, 1.023, 1.016, 1.007, 1.006. The first-run simulator updated the features every tick inside a 0.25 s bin; M2' is defined on features frozen at the bin start. Fixed by F2.

## 5. Fixes (approved Oct 29, 2026)

- **F1** (`sim.constants`): p_stop = lam_stop dt, p_resume = lam_resume dt, p_start = lam_start dt / (1 + lam_start dt).
- **F1b** (`motion.py`, against the locked P17 spec "stop exposure excludes time at the wall"): a moving step that lands on the wall is wall time, not stop exposure. P17 rerun on G-train with the same spec, seed and bootstrap (`p17_motion.ipynb`); the old table is `p17_rates_v1.csv`. lam_stop: lab v09 3.039 -> 3.259, old CL 1.760 -> 1.804, CC 10.350 -> 11.101; lam_resume and lam_start unchanged (CIs too). Both P17 verdicts unchanged: separate lam_start (3 of 3 disjoint), direction = coin flip (0.512, CI 0.483-0.540).
- **F2** (`sim.bin_hazard`): M2' / M3 reversals drawn per estimator bin, as `hazard.build_hazard_table` defines bins (0.25 s from the first move; positions from the last row strictly before the bin start, CC on the 0.1 s lab clock; time features at the bin start; no hazard in bins that start mid-event; the bin's decisions fire with 1 - (1 - h)^(1/n)). APEM stays per tick.
- **F3** (`sim.tick`): a reversal flips phi at once and is protected only while it is open on the detector; on the 0.1 grid this is the old 2-step script.
- Checked in `tests/test_sim.py`: in lab v09 the simulator's bins have exactly the table's features, kept / dropped status and y_rev bins; in old CL the features and status are exact; in CC >= 99% (the cloud logger's 1-2 ms row split).

## 6. Rerun of the recovery (once)

Fixed before the rerun:
- G6train design, L1, R = 2, seed 573, the code of section 5. Run once; the seed-574 rule of the first run does not apply.
- Pass rules as before: each M2' weight within 2 model SEs of the value the simulation uses (`p11_coefs.csv`); each lambda within its 95% CI in the **new** `p17_rates.csv`.
- Plus: M2' refit on the lab v09 rows only (all are lab = 1, so the design has no lab column and its intercept is compared with const + lab; the pooled lab weight is read against that intercept). Every weight must pass there.
- Reading: a pooled miss that disappears in the lab v09 refit = measurement dating, recorded as a limitation; a lab v09 miss = a real bug: stop and report.
- Output: `results/tables/p23_recovery.csv` (column `fit`: pooled, lab v09 only, P17 rates).

In [7]:
FEATS = ["A", "abs_s", "at_end", "time_at_end_log1p", "time_since_rev_log1p"]    # M2'
TRUE = sim.coefs("M2'")                                   # the weights the simulation uses
P17 = pd.read_csv(config.P17_RATES).set_index(["interface", "rate"])      # after F1b
G_TRAIN = {"bins": 336073, "reversals": 6073}              # G-train hazard table, all cells (P8, P11)


def weight_rows(model, fit_name, true):
    rows = []
    for term, b in model.params.items():
        se = model.bse[term]
        rows.append({"fit": fit_name, "part": "M2' weight", "parameter": term, "true": true[term], "recovered": b,
                     "SE": se, "z": (b - true[term]) / se, "ci_low": np.nan, "ci_high": np.nan,
                     "pass": abs(b - true[term]) <= 2 * se})
    return rows


def recover(seed, R=2):
    """Simulate L1 on G6train, rebuild the hazard table, refit M2' (pooled and lab v09 only), re-estimate the
    P17 rates. Returns (one row per parameter, counts)."""
    start = time.time()
    runs, table = design.simulate_design(designs["G6train"], "L1", R=R, seed=seed)
    ht, dropped = hazard.build_hazard_table(runs, b=0.25, m=2)     # CC on the 0.1 s lab clock
    pooled = fit.fit(ht, FEATS, False)                             # R0: shared weights + lab
    v09 = ht[ht["interface"] == "lab_v09"]
    X = fit.design(v09, FEATS, False).drop(columns="lab")          # lab = 1 on every lab v09 row
    only = sm.Logit(v09["y_rev"].astype(float), X).fit(disp=0)
    true09 = {**TRUE, "const": TRUE["const"] + TRUE["lab"]}
    rows = weight_rows(pooled, "pooled", TRUE) + weight_rows(only, "lab v09 only", true09)
    rows[len(TRUE)]["parameter"] = "const (+ lab)"
    rates = motion.rates_table(runs).set_index(["interface", "rate"])
    for (iface, rate), p in P17.loc[motion.INTERFACES].iterrows():
        r = rates.loc[(iface, rate)]
        se = np.sqrt(r["events"]) / r["exposure_s"]
        rows.append({"fit": "P17 rates", "part": "rate", "parameter": f"{iface} {rate}", "true": p["estimate"],
                     "recovered": r["estimate"], "SE": se, "z": (r["estimate"] - p["estimate"]) / se,
                     "ci_low": p["ci_low"], "ci_high": p["ci_high"], "pass": p["ci_low"] <= r["estimate"] <= p["ci_high"]})
    counts = {"seed": seed, "rounds": len(runs), "bins": len(ht), "reversals": int(ht["y_rev"].sum()),
              "lab v09 bins": len(v09), "dropped mid-event bins": len(dropped),
              "converged": bool(pooled.mle_retvals["converged"] and only.mle_retvals["converged"]),
              "minutes": round((time.time() - start) / 60, 1)}
    return pd.DataFrame(rows).assign(seed=seed), counts


rec, counts = recover(573)                                        # several minutes
rec.to_csv(config.P23_RECOVERY, index=False)
pd.DataFrame([counts])

,seed,rounds,bins,reversals,lab v09 bins,dropped mid-event bins,converged,minutes
0,573,488,1470492,36947,662685,17152,True,4.8


Simulated bins and reversals next to G-train's (per copy = total / R):

In [8]:
side = pd.DataFrame({"G-train (P8)": G_TRAIN, "simulated, seed 573, R = 2": {k: counts[k] for k in G_TRAIN},
                     "per copy": {k: counts[k] / 2 for k in G_TRAIN}}).T
side["reversals per 1,000 bins"] = 1000 * side["reversals"] / side["bins"]
side.round(1)

,bins,reversals,"reversals per 1,000 bins"
G-train (P8),336073.0,6073.0,18.1
"simulated, seed 573, R = 2",1470492.0,36947.0,25.1
per copy,735246.0,18473.5,25.1


Recovery table (`results/tables/p23_recovery.csv`):

In [9]:
with pd.option_context("display.float_format", "{:.4f}".format):
    display(rec[["fit", "parameter", "true", "recovered", "SE", "z", "ci_low", "ci_high", "pass"]])

,fit,parameter,true,recovered,SE,z,ci_low,ci_high,pass
0,pooled,const,-3.1941,-3.1307,0.0240,2.6400,NaN,NaN,False
1,pooled,lab,0.0578,0.0314,0.0111,-2.3751,NaN,NaN,False
2,pooled,A,-2.1074,-2.1152,0.0221,-0.3503,NaN,NaN,True
3,pooled,abs_s,1.0476,0.9374,0.0363,-3.0372,NaN,NaN,False
4,pooled,at_end,0.4446,0.5094,0.0239,2.7170,NaN,NaN,False
5,pooled,time_at_end_log1p,-0.3774,-0.3709,0.0108,0.6023,NaN,NaN,True
6,pooled,time_since_rev_log1p,-0.2093,-0.2149,0.0118,-0.4735,NaN,NaN,True
7,lab v09 only,const (+ lab),-3.1363,-3.1626,0.0361,-0.7276,NaN,NaN,True
8,lab v09 only,A,-2.1074,-2.1450,0.0329,-1.1437,NaN,NaN,True
9,lab v09 only,abs_s,1.0476,1.0869,0.0545,0.7193,NaN,NaN,True


In [10]:
for name, r in rec.groupby("fit", sort=False):
    print(f"{name}: {int(r['pass'].sum())} of {len(r)} pass ({'PASS' if r['pass'].all() else 'FAIL'})")
pooled = rec[rec["fit"] == "pooled"].set_index("parameter")["pass"]
only = rec[rec["fit"] == "lab v09 only"].set_index("parameter")["pass"].rename({"const (+ lab)": "const"})
w = pd.DataFrame({"pooled": pooled, "lab v09 only": only.reindex(pooled.index)})
w.loc["lab", "lab v09 only"] = only["const"]          # lab v09 has no lab column: its intercept is const + lab
w["reading"] = np.where(~w["lab v09 only"].astype(bool), "lab v09 miss: a real bug, stop and report",
                        np.where(~w["pooled"].astype(bool), "pooled miss only: measurement dating (limitation)", "pass"))
w

pooled: 3 of 7 pass (FAIL)
lab v09 only: 6 of 6 pass (PASS)
P17 rates: 9 of 9 pass (PASS)


,pooled,lab v09 only,reading
parameter,,,
const,False,True,pooled miss only: measurement dating (limitation)
lab,False,True,pooled miss only: measurement dating (limitation)
A,True,True,pass
abs_s,False,True,pooled miss only: measurement dating (limitation)
at_end,False,True,pooled miss only: measurement dating (limitation)
time_at_end_log1p,True,True,pass
time_since_rev_log1p,True,True,pass


**Pooled weights against the P12 bootstrap SE (descriptive only; no verdict changes).** P12 resampled G-train groups within EL, CL, CC (1,000 resamples) and refit M2'; its bootstrap SE is 1.2-3.5 times the G-train model SE (`results/tables/p12_beta_ci.csv`).

In [ ]:
p12 = pd.read_csv(config.P12_BETA_CI).set_index("quantity")
d = rec[rec["fit"] == "pooled"].set_index("parameter")[["true", "recovered", "SE"]].rename(columns={"SE": "model SE"})
d["P12 bootstrap SE"] = p12.loc[d.index, "boot_se"]
d["(recovered - true) / model SE"] = (d["recovered"] - d["true"]) / d["model SE"]
d["(recovered - true) / P12 SE"] = (d["recovered"] - d["true"]) / d["P12 bootstrap SE"]
d.round(4)

,true,recovered,model SE,P12 bootstrap SE,(recovered - true) / model SE,(recovered - true) / P12 SE
parameter,,,,,,
const,-3.1941,-3.1307,0.0240,0.0929,2.6400,0.6825
lab,0.0578,0.0314,0.0111,0.0866,-2.3751,-0.3053
A,-2.1074,-2.1152,0.0221,0.1673,-0.3503,-0.0464
abs_s,1.0476,0.9374,0.0363,0.1145,-3.0372,-0.9635
at_end,0.4446,0.5094,0.0239,0.0746,2.7170,0.8683
time_at_end_log1p,-0.3774,-0.3709,0.0108,0.0542,0.6023,0.1204
time_since_rev_log1p,-0.2093,-0.2149,0.0118,0.0757,-0.4735,-0.0741
